作用：对历史消息列表进行**摘要&总结**，达到**压缩上下文**的效果。

原理：在达到触发条件时，调用大模型对历史消息进行摘要，将摘要的结果作为HumanMessage，放到消息列表最开始的位置。

## 参数说明
> 注意：本节中间件的参数说明不保证包含完整参数列表，不常用或被标记为过时的参数被省略。

### 参数1：model —用于摘要的模型
可以是模型名称也可以是模型对象，如果传递的是模型名称，底层会调用 `init_chat_model` 初始化模型。

### 参数2：trigger —摘要触发条件
是一个列表，每个元素对应一个条件，当**任一条件满足**时，触发摘要。
1. `tokens`：token的数量，历史token的累计数量达到该值触发摘要。
2. `messages`：历史消息数量，历史消息条数达到该值触发摘要。
3. `fraction`：上下文长度比例。历史token的累计数量达到模型的 `max_input_tokens*fraction` 触发摘要

如果条件包含 `fraction`，要求模型的profile包含 `max_input_tokens`，Deepseek模型的profile为空，此时需要手动添加该配置项。Deepseek-V3.2的上下文长度为128K。

### 参数3：keep —摘要时保留的原始消息
支持三种条件，但和trigger不同，keep同一时间只接收一种条件。
1. `tokens`：摘要时保留的token数量。
2. `messages`：摘要时保留的历史消息条数。
3. `fraction`：摘要时保留 `max_input_tokens*fraction` 个token。

### 参数4：token_counter —统计token数量的函数
默认使用LangChain提供的 `count_tokens_approximately`，一般不用更改。
> 对于纯文本消息，该函数的大致思路是先统计消息的字符数，也就是 len(字符串)，然后再除以每个token大致的字符数，转换为粗略的token数，再加一些额外开销，作为估算的token数。

### 参数5：summary_prompt —摘要时的自定义提示词
该提示词需要包含 `{messages}` 占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。

### 参数6：trim_token_to_summarize —摘要时历史消息的最大token数
如果历史消息token数大于该值，则会被裁剪。默认为 `"4000"`。
如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。

## 示例1

测试trigger、keep参数

下面代码有一个细节 如果供应商没有提供model_profile，需要自定义custom_profile字典格式

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

custom_profile = {
    "max_input_tokens": 128_000
}

model_ds = init_chat_model(
    model="deepseek-v4-flash",
   # model_provider="deepseek",
    profile=custom_profile,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

model_gpt = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

In [5]:
# 摘要中间件使用代码
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model=model_ds,
    middleware=[
        SummarizationMiddleware(
            model=model_ds,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT

用户（老王）与AI助手（小王）初次打招呼并相互介绍，建立友好互动关系。

## SUMMARY

- 用户自称“老王”，AI助手自我介绍为“小王”。
- 双方进行了一次简单的友好问候，用户表示很高兴认识小王。
- 系统预设AI为非常友好的助手，实际回复符合此设定。
- 没有深入的任务或具体问题被提出，属于初次接触破冰阶段。

## ARTIFACTS

None

## NEXT STEPS

- 用户尚未提出任何具体请求或任务，后续可等待用户进一步指示或主动询问用户需求。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王您别误会！小王刚才的意思是：看到您这么热情地打招呼，我也特别开心，但一想您还没给我下达任务呢，我得接着努力表现才行。完全没有任何冒犯的意思，倒是我嘴笨，让您见笑了。

您放心，我态度是绝对端正的。既然您来了，那咱们就正式聊起来吧——老王您今天想聊点啥？或者有什么事需要小王帮忙跑跑腿、动动脑的？您尽管吩咐！


## 示例2

试summary_prompt参数，其中{messages}占位符是历史消息列表，必须包含

自定义摘要的提示模板，可以是一句话，也可以自定义结构化摘要

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

custom_profile = {
    "max_input_tokens": 1_000_000
}

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    profile=custom_profile,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

messages = [
    SystemMessage("你是个非常有好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思，你是谁？")
]

agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.0001)
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要如下：

- 系统设定：我是一个非常友好的 AI 助手。
- 用户自我介绍：用户说自己是老王，并询问我是谁。
- 我的回复：我回答自己是“小王”。
- 后续交流：用户表示很高兴认识我。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思，你是谁？
================================== Ai Message ==================================

哈哈，老王别紧张，我开玩笑呢！我是小王，你的AI助手，很高兴认识你！刚才那句话是想逗你玩一下，别往心里去哈~有什么需要帮忙的尽管说！
